In [ ]:
from typing import Union, Dict, Any
import numpy as np
import pandas as pd

# CAPM utilities for a Jupyter cell (new file)
# GitHub Copilot

import statsmodels.api as sm


def _to_series(x: Union[pd.Series, pd.DataFrame, np.ndarray, list, float], length: int = None, name: str = None) -> pd.Series:
    if isinstance(x, pd.Series):
        return x.rename(name) if name else x
    if isinstance(x, pd.DataFrame):
        if x.shape[1] != 1:
            raise ValueError("DataFrame must have a single column")
        s = x.iloc[:, 0].copy()
        s.name = name or s.name
        return s
    arr = np.array(x)
    if arr.ndim == 0:  # scalar
        if length is None:
            return pd.Series([float(arr)], name=name)
        return pd.Series([float(arr)] * length, name=name)
    if arr.ndim == 1:
        return pd.Series(arr, name=name)
    raise ValueError("Unsupported input shape")


def capm_regression(
    asset_returns: Union[pd.Series, np.ndarray, list],
    market_returns: Union[pd.Series, np.ndarray, list],
    risk_free_rate: Union[float, pd.Series, np.ndarray, list] = 0.0,
) -> Dict[str, Any]:
    """
    Esegue la regressione CAPM:
      (R_asset - R_f) = alpha + beta * (R_market - R_f) + eps

    Restituisce: alpha, beta, SE, t-stats, p-values, R-squared, attesi CAPM, residui, predetti.
    Input can be scalars or series/arrays. Aligns lengths; if risk_free_rate is scalar it is broadcast.
    """
    a = _to_series(asset_returns, name="asset")
    m = _to_series(market_returns, name="market")
    n = max(len(a), len(m))
    if len(a) != len(m):
        # try to align by index if both are series with indexes, else raise
        if isinstance(a, pd.Series) and isinstance(m, pd.Series):
            df = pd.concat([a, m], axis=1).dropna()
            a = df.iloc[:, 0]
            m = df.iloc[:, 1]
        else:
            raise ValueError("asset_returns and market_returns must have same length or be index-alignable Series")
    rf = _to_series(risk_free_rate, length=len(a), name="rf")
    if len(rf) != len(a):
        # try alignment by index
        if isinstance(rf, pd.Series) and isinstance(a, pd.Series):
            combined = pd.concat([a, m, rf], axis=1).dropna()
            a, m, rf = combined.iloc[:, 0], combined.iloc[:, 1], combined.iloc[:, 2]
        else:
            # broadcast scalar handled in _to_series; otherwise mismatch
            raise ValueError("risk_free_rate length mismatch")

    # excess returns
    y = a - rf
    x = m - rf
    X = sm.add_constant(x)  # constant = alpha
    model = sm.OLS(y, X, missing="drop")
    res = model.fit()

    alpha = res.params["const"]
    beta = res.params[x.name if x.name is not None else x.index.name or 1] if "const" in res.params.index else res.params[0]
    # More robust retrieval:
    if "const" in res.params.index:
        beta = res.params.drop("const").iloc[0]
    else:
        beta = res.params.iloc[0]

    results = {
        "alpha": float(alpha),
        "beta": float(beta),
        "alpha_se": float(res.bse["const"]) if "const" in res.bse.index else float(res.bse.iloc[0]),
        "beta_se": float(res.bse.drop("const").iloc[0]) if "const" in res.bse.index else float(res.bse.bse[0]),
        "alpha_t": float(res.tvalues["const"]) if "const" in res.tvalues.index else float(res.tvalues.iloc[0]),
        "beta_t": float(res.tvalues.drop("const").iloc[0]) if "const" in res.tvalues.index else float(res.tvalues.tvalues[0]),
        "alpha_pvalue": float(res.pvalues["const"]) if "const" in res.pvalues.index else float(res.pvalues.iloc[0]),
        "beta_pvalue": float(res.pvalues.drop("const").iloc[0]) if "const" in res.pvalues.index else float(res.pvalues.pvalues[0]),
        "rsquared": float(res.rsquared),
        "nobs": int(res.nobs),
        "fitted_values": res.fittedvalues,
        "residuals": res.resid,
        "model_summary": res.summary(),  # statsmodels summary object
    }

    # expected CAPM return using sample mean of market (can be replaced with user-specified expected market return)
    mean_market = float(m.mean())
    mean_rf = float(rf.mean())
    expected_capm = mean_rf + beta * (mean_market - mean_rf)
    results["expected_return_capm"] = float(expected_capm)
    results["mean_market"] = mean_market
    results["mean_rf"] = mean_rf

    return results


def capm_expected_return(beta: float, risk_free_rate: float, expected_market_return: float) -> float:
    """
    Calcola E[R_i] = R_f + beta * (E[R_m] - R_f)
    """
    return float(risk_free_rate + beta * (expected_market_return - risk_free_rate))


if __name__ == "__main__":
    # Esempio minimo d'uso (puoi rimuovere quando usi la cella in notebook)
    import matplotlib.pyplot as plt

    # dati sintetici d'esempio
    np.random.seed(42)
    n = 250
    rf_series = pd.Series(0.0002, index=range(n))  # e.g. daily rf ~ 0.02% per day
    market = pd.Series(np.random.normal(0.0005, 0.01, size=n))
    # asset con beta ~ 1.3, alpha 0.0001
    asset = 0.0001 + 1.3 * market + np.random.normal(0, 0.01, size=n)

    res = capm_regression(asset, market, rf_series)
    print("alpha:", res["alpha"])
    print("beta :", res["beta"])
    print("R^2  :", res["rsquared"])
    print("Expected CAPM return (sample):", res["expected_return_capm"])

    # Plot asset excess vs market excess and fit line
    y = pd.Series(asset) - rf_series
    x = pd.Series(market) - rf_series
    plt.scatter(x, y, alpha=0.6, label="excess returns")
    b = res["beta"]
    a = res["alpha"]
    xs = np.linspace(x.min(), x.max(), 100)
    plt.plot(xs, a + b * xs, color="red", label=f"CAPM fit (alpha={a:.4f}, beta={b:.2f})")
    plt.xlabel("Market excess return")
    plt.ylabel("Asset excess return")
    plt.legend()
    plt.show()